## Code to check configurations of points 

given four points $u,v,w,x$ in the plane we want to know when we can separate $u,v,w$ and $x$ by a line $L$ of positive slope such that either  

- $x$ pushes up and $u,v,w$ push right 
- $x$ pushes right and $u,v,w$ push up 

In [2]:
import gudhi
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib
from math import cos, sin, pi, atan

import numpy.linalg

## for checking if in convex hull
import scipy.spatial
from scipy.optimize import linprog


import sys 
sys.path.append('../Code/')
import modules as mod

# This reloads external modules every time a cell is run
# so we don't have to reboot kernel every time we modify classes.py
%load_ext autoreload
%autoreload 2

In [3]:
# here we define four critical values to check. 
# later we'll want to import these 4 at a time from a bigger set, but for now we just hardcode them.
c1 = mod.critical_val((0,0), 0)
c2 = mod.critical_val((0,1), 0)
c3 = mod.critical_val((1,0),0)
c4 = mod.critical_val((1,1),0)
CritPts = [c1,c2,c3,c4]


In [4]:
c1.coords

(0, 0)

## Checking if a point $x$ is in the convex hull of a set of points in $\mathbb{R}^2$

* being in the convex hull includes the boundary of the convexhull
* in_hull_old is the code from stack exchange 
* in_hull is modified to fit our case


In [5]:
def in_hull_old(points, x):
    n_points = len(points)
    n_dim = len(x)
    c = np.zeros(n_points)
    A = np.r_[points.T,np.ones((1,n_points))]
    b = np.r_[x, np.ones(1)]
    lp = linprog(c, A_eq=A, b_eq=b)
    return lp.success


def in_hull(points, x):
    ## check if x in convex hull of points 
    ## points = array of pts in R^2 
    
    n_points = len(points)
    c = np.zeros(n_points)
    A = np.r_[points.T,np.ones((1,n_points))]
    b = np.r_[x, np.ones(1)]
    lp = linprog(c, A_eq=A, b_eq=b, )
    return lp.success



In [6]:
n_points = 3
n_dim = 2
Z = np.random.rand(n_points,n_dim)
x = np.random.rand(n_dim)
print(in_hull(Z, x))

False


In [7]:
points = np.array([[0,0],[1,0],[1,1]])
x = (0.5,0)
in_hull(points, x)

True

In [8]:
points

array([[0, 0],
       [1, 0],
       [1, 1]])

## 1. the case where $x$ pushes up and $u,v,w$ push right
* the sort function returns the rightmost and bottommost of the points $u,v,w$ allowing us to create the boudary of the convex hull along with the region top left of the convex hull
* check_in_quadrant_tl returns true if $x$ is in the top left quadrant defined by and containing points $p,q$
* check_x_push_up tells us if we can find a line such that $x$ pushes up while $u,v,w$ push right. This excludes the cases where $x$ is on the boundary of the convex hull of $u,v,w$ 

In [9]:
def sort_bottom_and_right(u,v,w): 
    """return the rightmost and bottommost points of u,v,w """
    
    rm = u
    if v[0]>rm[0]:
        rm = v
    if w[0] > rm[0]:
        rm = w
        
    bm = u
    if v[1]< bm[1]:
        bm = v
    if w[1] < bm[1]:
        bm = w

    return bm, rm


def check_in_quadrant_tl(x, p, q): 
    """"assume p is the bottomost pt of u,v,w
        assume q is the rightmost of u,v,w 
        return True if x is in the top left quadrant 
    """
 
    if x[0]<= q[0] and x[1]>=p[1]: 
        return True 
    else: 
        return False 
        

In [10]:
def check_x_push_up(x,u,v,w): ## Check-point-A and Check-point-B like in the overleaf 
    """Check if the points x,u,v,w are admissible for:
            - x pushing up
            - u,v,w pushing right 
        returns True if they are an admissible quadruple 
        and False if they are not admissible or would create superfluous omegapoints 
    """    
    p,q = sort_bottom_and_right(u,v,w)
    a = (q[0],p[1])
    
    ## First check if x is in the upper left quadrant: 
    if check_in_quadrant_tl(x,p,q): 
        ## Check if x is in convex hull
        if in_hull(np.array([u,v,w]),x):
            return False
        
        elif in_hull(np.array[u,v,w,a],x ):
            return True   
    else: 
        return True 

In [11]:
p1= (0,1)
p2 = (0,2)
p3= (4,5)
points = np.array([p1,p2,p3])

In [12]:
points

array([[0, 1],
       [0, 2],
       [4, 5]])

## 2. the case where $x$ pushes right and $u,v,w$ push up
* the sort function returns the rightmost and bottommost of the points $u,v,w$ allowing us to create the boudary of the convex hull along with the region bottom right of the convex hull
* check_in_quadrant_br returns true if $x$ is in the bottom right quadrant defined by and containing points $p,q$
* check_x_push_right tells us if we can find a line such that $x$ pushes right while $u,v,w$ push up. This excludes the cases where $x$ is on the boundary of the convex hull of $u,v,w$ 

In [13]:
def sort_top_and_left(u,v,w): 
    """return the leftmost and topmost points of u,v,w """
        
    lm = u
    if v[0]<lm[0]:
        lm = v
    if w[0] < lm[0]:
        lm = w
        
    tm = u
    if v[1]> tm[1]:
        tm = v
    if w[1] > tm[1]:
        tm = w

    return tm, lm


def check_in_quadrant_br(x, p, q): 
    """"assume p is the topmost pt of u,v,w
        assume q is the leftmost of u,v,w 
        return True if x is in the bottom right quadrant(including the boundary)
    """
    
    if x[0]>= q[0] and x[1]<=p[1]: 
        return True 
    else: 
        return False 

In [14]:
def check_x_push_right(x,u,v,w): 
    """Check if the points x,u,v,w are admissible for:
            - x pushing right
            - u,v,w pushing up 
        returns True if they are an admissible quadruple 
        and False if they are not admissible or would create superfluous omegapoints 
    """    
    
    p,q = sort_top_and_left(u,v,w)
    a = (q[0],p[1])
    
    ## First check if x is in the upper left quadrant: 
    if check_in_quadrant_br(x,p,q): 
        ## Check if x is in convex hull
        if in_hull(np.array([u,v,w]),x):
            return False
        
        elif in_hull(np.array[u,v,w,a],x ):
            return True   
    else: 
        return True 
    
    

## check on small examples
* first check if code works properly
* example with 4 or 8 points from other notebook? how many do we discard? 
* check lines 34 and 51 in algo 3 overleaf --> some quadruples will genereate only onne omega pt not two: modify in implementation 

#### Example 1

In [15]:
x1 =
x2 = 
x3 = 
x4 = 
u = 
v = 
w = 

SyntaxError: invalid syntax (<ipython-input-15-fd3d2eb7eba1>, line 1)

#### Example 2


In [ ]:
x1 =
x2 = 
x3 = 
x4 = 
u = 
v = 
w = 

#### Example 3